In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/__results__.html
/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/aml_xgb_model.pkl
/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/__notebook__.ipynb
/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/__output__.json
/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/threshold_evaluation_xgb.png
/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/threshold_evaluation_lr.png
/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/custom.css
/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/__results___files/__results___25_0.png
/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/__results___files/__results___19_0.png
/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/__results___files/__results___35_0.png
/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/__results___files/_

## Creating the Cumulative State and Recent Transaction State

In [3]:
import pandas as pd
import numpy as np

from collections import defaultdict, deque
from dataclasses import dataclass, field


# =========================================================
# CONFIG
# =========================================================

REQUIRED_COLUMNS = [
    "Datetime",
    "Sender_account",
    "Receiver_account",
    "Amount",
]

WINDOW_1H = pd.Timedelta(hours=1)
WINDOW_24H = pd.Timedelta(hours=24)


# =========================================================
# 1. CUMULATIVE STATE
# =========================================================

def build_cumulative_state(historical_df: pd.DataFrame):
    """
    Build lifetime/cumulative state from historical transactions.

    Returns:
        sender_state: DataFrame indexed by Sender_account
        receiver_state: DataFrame indexed by Receiver_account

        sender_seen_receivers: dict[account -> set(receiver)]
        receiver_seen_senders: dict[account -> set(sender)]
    """

    df = historical_df.copy()

    # -----------------------------
    # Validation
    # -----------------------------
    missing_cols = [
        col for col in REQUIRED_COLUMNS
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Missing required columns: {missing_cols}"
        )

    # -----------------------------
    # Preparation
    # -----------------------------
    df["Datetime"] = pd.to_datetime(df["Datetime"])

    df = df.sort_values("Datetime").reset_index(drop=True)

    # =====================================================
    # SENDER STATE
    # =====================================================

    sender_state = (
        df.groupby("Sender_account")
        .agg(
            sender_tx_count_history=(
                "Amount",
                "count"
            ),

            sender_total_amount_history=(
                "Amount",
                "sum"
            ),

            sender_avg_amount_history=(
                "Amount",
                "mean"
            ),

            sender_unique_receivers_history=(
                "Receiver_account",
                "nunique"
            ),

            sender_time_since_last_tx=(
                "Datetime",
                "max"
            ),
        )
    )

    # =====================================================
    # RECEIVER STATE
    # =====================================================

    receiver_state = (
        df.groupby("Receiver_account")
        .agg(
            receiver_tx_count_history=(
                "Amount",
                "count"
            ),

            receiver_total_amount_history=(
                "Amount",
                "sum"
            ),

            receiver_avg_amount_history=(
                "Amount",
                "mean"
            ),

            receiver_unique_senders_history=(
                "Sender_account",
                "nunique"
            ),

            receiver_time_since_last_tx=(
                "Datetime",
                "max"
            ),
        )
    )

    # Tambahkan ini di akhir fungsi build_cumulative_state()
    pair_state = (
    df.groupby(["Sender_account", "Receiver_account"])
    .size()
    .to_dict()  # Menghasilkan dict: {(sender, receiver): count}
    )

    # =====================================================
    # UNIQUE COUNTERPARTY SET
    #
    # Needed later when updating the state.
    # =====================================================

    sender_seen_receivers = (
        df.groupby("Sender_account")["Receiver_account"]
        .agg(lambda x: set(x))
        .to_dict()
    )

    receiver_seen_senders = (
        df.groupby("Receiver_account")["Sender_account"]
        .agg(lambda x: set(x))
        .to_dict()
    )

    return (
        sender_state,
        receiver_state,
        sender_seen_receivers,
        receiver_seen_senders,
        pair_state
    )


# =========================================================
# 2. RECENT TRANSACTION STATE
# =========================================================

@dataclass
class WindowState:
    """
    Stores transactions for one entity
    for 1-hour and 24-hour windows.
    """

    transactions_1h: deque = field(default_factory=deque)
    transactions_24h: deque = field(default_factory=deque)

    total_1h: float = 0.0
    total_24h: float = 0.0


class RecentTransactionState:
    """
    Stateful store for sliding-window features.

    Separate state is maintained for:
        - sender
        - receiver

    Each entity stores only recent transactions.
    """

    def __init__(self):

        self.sender = defaultdict(WindowState)
        self.receiver = defaultdict(WindowState)

        self.current_time = None

    # =====================================================
    # INTERNAL CLEANUP
    # =====================================================

    def _cleanup_entity(
        self,
        state: WindowState,
        current_time: pd.Timestamp
    ):
        """
        Remove transactions that are outside 1h / 24h windows.
        """

        cutoff_1h = current_time - WINDOW_1H
        cutoff_24h = current_time - WINDOW_24H

        # -------------------------
        # Remove expired 1h
        # -------------------------

        while (
            state.transactions_1h
            and state.transactions_1h[0][0] <= cutoff_1h
        ):
            _, amount = state.transactions_1h.popleft()

            state.total_1h -= amount

        # -------------------------
        # Remove expired 24h
        # -------------------------

        while (
            state.transactions_24h
            and state.transactions_24h[0][0] <= cutoff_24h
        ):
            _, amount = state.transactions_24h.popleft()

            state.total_24h -= amount

    # =====================================================
    # ADD TRANSACTION
    # =====================================================

    def add_transaction(
        self,
        entity_type: str,
        entity_id,
        dt: pd.Timestamp,
        amount: float
    ):
        """
        Add one transaction AFTER prediction.

        entity_type:
            'sender'
            'receiver'
        """

        if entity_type == "sender":
            state = self.sender[entity_id]

        elif entity_type == "receiver":
            state = self.receiver[entity_id]

        else:
            raise ValueError(
                "entity_type must be 'sender' or 'receiver'"
            )

        # Clean expired transactions first
        self._cleanup_entity(
            state,
            dt
        )

        # --------------------------------
        # Add to 24h window
        # --------------------------------

        state.transactions_24h.append(
            (dt, float(amount))
        )

        state.total_24h += float(amount)

        # --------------------------------
        # Add to 1h window
        # --------------------------------

        state.transactions_1h.append(
            (dt, float(amount))
        )

        state.total_1h += float(amount)

    # =====================================================
    # GET WINDOW FEATURES
    # =====================================================

    def get_features(
        self,
        entity_type: str,
        entity_id,
        current_time: pd.Timestamp
    ):
        """
        Get 1h / 24h window aggregates BEFORE
        processing the current transaction.
        """

        if entity_type == "sender":
            state = self.sender[entity_id]

        elif entity_type == "receiver":
            state = self.receiver[entity_id]

        else:
            raise ValueError(
                "entity_type must be 'sender' or 'receiver'"
            )

        # Remove expired transactions
        self._cleanup_entity(
            state,
            current_time
        )

        # --------------------------------
        # 1 hour
        # --------------------------------

        count_1h = len(
            state.transactions_1h
        )

        total_1h = state.total_1h

        avg_1h = (
            total_1h / count_1h
            if count_1h > 0
            else 0.0
        )

        # --------------------------------
        # 24 hour
        # --------------------------------

        count_24h = len(
            state.transactions_24h
        )

        total_24h = state.total_24h

        avg_24h = (
            total_24h / count_24h
            if count_24h > 0
            else 0.0
        )

        return {
            "count_1h": count_1h,
            "total_amount_1h": total_1h,
            "avg_amount_1h": avg_1h,

            "count_24h": count_24h,
            "total_amount_24h": total_24h,
            "avg_amount_24h": avg_24h,
        }

In [4]:
def build_recent_transaction_state(
    historical_df: pd.DataFrame
):
    """
    Initialize recent transaction state using transactions
    from the last 24 hours of historical_df.
    """

    df = historical_df.copy()

    df["Datetime"] = pd.to_datetime(df["Datetime"])

    df = (
        df.sort_values("Datetime")
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            "historical_df is empty."
        )

    latest_time = df["Datetime"].max()

    cutoff_24h = (
        latest_time - WINDOW_24H
    )

    # -----------------------------------------
    # Only keep transactions relevant to
    # current 24-hour state
    # -----------------------------------------

    recent_df = df[
        df["Datetime"] > cutoff_24h
    ].copy()

    recent_state = RecentTransactionState()

    # -----------------------------------------
    # Build state
    # -----------------------------------------

    for row in recent_df.itertuples(index=False):

        dt = row.Datetime
        sender = row.Sender_account
        receiver = row.Receiver_account
        amount = float(row.Amount)

        recent_state.add_transaction(
            entity_type="sender",
            entity_id=sender,
            dt=dt,
            amount=amount,
        )

        recent_state.add_transaction(
            entity_type="receiver",
            entity_id=receiver,
            dt=dt,
            amount=amount,
        )

    recent_state.current_time = latest_time

    return recent_state

## Load Historical Data to Build State

In [5]:
historical_df = pd.read_parquet("/kaggle/input/notebooks/mrajareivan/data-understanding-aml/AML_data_from_eda.parquet")
historical_df.head()

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type,Datetime,is_cross_border,is_currency_conversion,hour,day_of_week,date_only,day_of_month
0,1900-01-01 10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits,2022-10-07 10:35:19,0,0,10,4,2022-10-07,7
1,1900-01-01 10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out,2022-10-07 10:35:20,1,1,10,4,2022-10-07,7
2,1900-01-01 10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Small_Fan_Out,2022-10-07 10:35:20,0,0,10,4,2022-10-07,7
3,1900-01-01 10:35:21,2022-10-07,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_In,2022-10-07 10:35:21,0,0,10,4,2022-10-07,7
4,1900-01-01 10:35:21,2022-10-07,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits,2022-10-07 10:35:21,0,0,10,4,2022-10-07,7


In [6]:
n = len(historical_df)
train_end = int(n * 0.85)

dev_df = historical_df.iloc[:train_end].copy()
test_df = historical_df.iloc[train_end:].copy()

print("dev:", len(dev_df), "test:", len(test_df))
print("dev end:", dev_df["Datetime"].max())
print("test start:", test_df["Datetime"].min()) ## --> dipakai untuk mensimulasikan inference

dev: 8079124 test: 1425728
dev end: 2023-07-05 15:24:39
test start: 2023-07-05 15:24:42


In [7]:
if pd.api.types.is_datetime64_any_dtype(historical_df['Datetime']):
    print('data type of the Datetime column is datetime')
    print("monotic increaing for datetime :", historical_df['Datetime'].is_monotonic_increasing)
else:
    historical_df["Datetime"] = pd.to_datetime(historical_df["Datetime"])
    historical_df = historical_df.sort_values("Datetime")

data type of the Datetime column is datetime
monotic increaing for datetime : True


In [8]:
(
    sender_state,
    receiver_state,
    sender_seen_receivers,
    receiver_seen_senders,
    pair_state
) = build_cumulative_state(
    dev_df
)

recent_state = build_recent_transaction_state(
    dev_df
)

## create synthetic data from the state

In [9]:
CATEGORICAL_COLUMNS = [
    "is_cross_border",
    "is_currency_conversion",
    "Payment_currency",
    "Received_currency",
    "Sender_bank_location",
    "Receiver_bank_location",
    "Payment_type",
]


def extract_categorical_values(historical_df):
    """
    Extract unique categorical values ONCE from historical_df.

    The returned sets can then be passed to AMLSyntheticGenerator.
    historical_df is NOT needed during synthetic generation.
    """

    categorical_values = {
        col: set(historical_df[col].dropna().unique())
        for col in CATEGORICAL_COLUMNS
    }

    return categorical_values

def extract_categorical_values_and_its_dist(historical_df):
    """Mengekstrak probabilitas nilai kategorikal SATU KALI dari historical_df.

    Output berupa dict dari {kolom: {nilai: probabilitas}}. historical_df
    TIDAK dibutuhkan lagi saat pembuatan data sintetik.
    """
    categorical_values = {}
    for col in CATEGORICAL_COLUMNS:
        # Menghitung proporsi/probabilitas tiap nilai (total = 1.0)
        prob_dict = (
            historical_df[col].dropna().value_counts(normalize=True).to_dict()
        )
        categorical_values[col] = prob_dict

    return categorical_values

categorical_values = extract_categorical_values(historical_df)

categorical_dist_values = extract_categorical_values_and_its_dist(historical_df)

## Save important artefacts

In [12]:
import joblib

ARTIFACT_DIR = "/kaggle/working/"

# ============================================================
# 1. INITIAL INFERENCE STATE
# ============================================================

initial_state = {
    "sender_state": sender_state,
    "receiver_state": receiver_state,

    "sender_seen_receivers": sender_seen_receivers,
    "receiver_seen_senders": receiver_seen_senders,

    "recent_state": recent_state,

    "pair_state": pair_state,
    "categorical_features_dict": categorical_values,
}

state_path = "initial_state.pkl"

joblib.dump(initial_state, state_path)


# ============================================================
# 2. TEST TRANSACTIONS
# ============================================================

test_path = "test_transactions.parquet"

test_df.to_parquet(test_path, index=False)


# ============================================================
# CHECK
# ============================================================

print("Artifacts successfully exported:")
print(f"1. {state_path}")
print(f"2. {test_path}")

print("\nFile sizes:")

for filename in os.listdir(ARTIFACT_DIR):
    path = os.path.join(ARTIFACT_DIR, filename)
    size_mb = os.path.getsize(path) / (1024 ** 2)

    print(
        f"{filename:<30} "
        f"{size_mb:.2f} MB"
    )

Artifacts successfully exported:
1. initial_state.pkl
2. test_transactions.parquet

File sizes:
initial_state.pkl              76.15 MB
.virtual_documents             0.00 MB
test_transactions.parquet      33.44 MB


# Prepare Feature


In [13]:
import joblib
import numpy as np
import pandas as pd


class AMLStatefulInference:

    def __init__(
        self,
        model_path,
        sender_state,
        receiver_state,
        sender_seen_receivers,
        receiver_seen_senders,
        recent_state,
        pair_state=None,
        categorical_features_dict=None,  # Opsional: jika ingin menyimpan dictionary validasi kategori
    ):
        # =====================================================
        # MODEL & METADATA LOADING
        # =====================================================
        bundle = joblib.load(model_path)

        self.pipeline = bundle["pipeline"]
        self.threshold = bundle.get("threshold", 0.5)

        self.numeric_features = bundle.get("numeric_features", [])
        self.categorical_features = bundle.get("categorical_features", [
            "is_cross_border",
            "is_currency_conversion",
            "Payment_currency",
            "Received_currency",
            "Sender_bank_location",
            "Receiver_bank_location",
            "Payment_type",
        ])
        self.feature_cols = bundle["feature_cols"]

        self.target_col = bundle.get("target_col")
        self.model_type = bundle.get("model_type")
        self.random_state = bundle.get("random_state")

        # Dictionary kategori valid (jika disediakan)
        self.categorical_features_dict = categorical_features_dict or bundle.get("categorical_features_dict", {})

        # =====================================================
        # STATE MANAGEMENT
        # =====================================================
        self.sender_state = sender_state
        self.receiver_state = receiver_state

        self.sender_seen_receivers = sender_seen_receivers
        self.receiver_seen_senders = receiver_seen_senders

        self.recent_state = recent_state
        self.pair_state = pair_state if pair_state is not None else {}
        # Posisi transaksi berikutnya yang akan diprediksi
        self.inference_position = 0

    def info(self):
        print(f"model threshold : {self.threshold}")
        print(f"model type : {self.model_type}")
        
    def reset_inference(self):
        self.inference_position = 0
        print("Inference cursor reset ke 0.")

    
    # =====================================================
    # CUMULATIVE FEATURES
    # =====================================================

    def _get_sender_cumulative_features(
        self,
        sender,
        current_time,
        amount,
    ):
        if sender in self.sender_state.index:
            state = self.sender_state.loc[sender]

            tx_count = state["sender_tx_count_history"]
            total_amount = state["sender_total_amount_history"]
            avg_amount = state["sender_avg_amount_history"]
            unique_receivers = state["sender_unique_receivers_history"]
            last_tx = state["sender_time_since_last_tx"]
        else:
            tx_count = 0
            total_amount = 0.0
            avg_amount = 0.0
            unique_receivers = 0
            last_tx = pd.NaT

        amount_vs_history_ratio = (
            amount / avg_amount if avg_amount > 0 else 0.0
        )

        if pd.isna(last_tx):
            time_since_last_tx = 0.0
        else:
            time_since_last_tx = (current_time - last_tx).total_seconds()

        return {
            "sender_tx_count_history": tx_count,
            "sender_total_amount_history": total_amount,
            "sender_avg_amount_history": avg_amount,
            "sender_amount_vs_history_ratio": amount_vs_history_ratio,
            "sender_unique_receivers_history": unique_receivers,
            "sender_time_since_last_tx": time_since_last_tx,
        }

    def _get_receiver_cumulative_features(
        self,
        receiver,
        current_time,
        amount,
    ):
        if receiver in self.receiver_state.index:
            state = self.receiver_state.loc[receiver]

            tx_count = state["receiver_tx_count_history"]
            total_amount = state["receiver_total_amount_history"]
            avg_amount = state["receiver_avg_amount_history"]
            unique_senders = state["receiver_unique_senders_history"]
            last_tx = state["receiver_time_since_last_tx"]
        else:
            tx_count = 0
            total_amount = 0.0
            avg_amount = 0.0
            unique_senders = 0
            last_tx = pd.NaT

        amount_vs_history_ratio = (
            amount / avg_amount if avg_amount > 0 else 0.0
        )

        if pd.isna(last_tx):
            time_since_last_tx = 0.0
        else:
            time_since_last_tx = (current_time - last_tx).total_seconds()

        return {
            "receiver_tx_count_history": tx_count,
            "receiver_total_amount_history": total_amount,
            "receiver_avg_amount_history": avg_amount,
            "receiver_amount_vs_history_ratio": amount_vs_history_ratio,
            "receiver_unique_senders_history": unique_senders,
            "receiver_time_since_last_tx": time_since_last_tx,
        }

    def _get_pair_cumulative_features(self, sender, receiver):
        pair_count = self.pair_state.get((sender, receiver), 0)
        return {
            "pair_tx_count_history": pair_count,
        }

    # =====================================================
    # WINDOWED FEATURES
    # =====================================================

    def _get_window_features(
        self,
        sender,
        receiver,
        current_time,
        amount,
    ):
        sender_window = self.recent_state.get_features(
            entity_type="sender",
            entity_id=sender,
            current_time=current_time,
        )

        receiver_window = self.recent_state.get_features(
            entity_type="receiver",
            entity_id=receiver,
            current_time=current_time,
        )

        s_avg_1h = sender_window["avg_amount_1h"]
        s_avg_24h = sender_window["avg_amount_24h"]
        r_avg_1h = receiver_window["avg_amount_1h"]
        r_avg_24h = receiver_window["avg_amount_24h"]

        sender_amount_vs_1h_avg = amount / s_avg_1h if s_avg_1h > 0 else 0.0
        sender_amount_vs_24h_avg = amount / s_avg_24h if s_avg_24h > 0 else 0.0
        receiver_amount_vs_1h_avg = amount / r_avg_1h if r_avg_1h > 0 else 0.0
        receiver_amount_vs_24h_avg = (
            amount / r_avg_24h if r_avg_24h > 0 else 0.0
        )

        return {
            "sender_tx_count_1h": sender_window["count_1h"],
            "sender_total_amount_1h": sender_window["total_amount_1h"],
            "sender_avg_amount_1h": s_avg_1h,
            "sender_tx_count_24h": sender_window["count_24h"],
            "sender_total_amount_24h": sender_window["total_amount_24h"],
            "sender_avg_amount_24h": s_avg_24h,
            "sender_amount_vs_1h_avg": sender_amount_vs_1h_avg,
            "sender_amount_vs_24h_avg": sender_amount_vs_24h_avg,
            "receiver_tx_count_1h": receiver_window["count_1h"],
            "receiver_total_amount_1h": receiver_window["total_amount_1h"],
            "receiver_avg_amount_1h": r_avg_1h,
            "receiver_tx_count_24h": receiver_window["count_24h"],
            "receiver_total_amount_24h": receiver_window["total_amount_24h"],
            "receiver_avg_amount_24h": r_avg_24h,
            "receiver_amount_vs_1h_avg": receiver_amount_vs_1h_avg,
            "receiver_amount_vs_24h_avg": receiver_amount_vs_24h_avg,
        }

    # =====================================================
    # BUILD ONE TRANSACTION'S FEATURES
    # =====================================================

    def build_features_for_transaction(self, row):
        dt = pd.to_datetime(row["Datetime"])
        sender = row["Sender_account"]
        receiver = row["Receiver_account"]
        amount = float(row["Amount"])

        # 1. CATEGORICAL FEATURES (Ekstraksi dari transaksi saat ini)
        cat_features = {}
        for col in self.categorical_features:
            if col in row:
                cat_features[col] = row[col]
            else:
                cat_features[col] = None

        # 2. CUMULATIVE FEATURES
        sender_features = self._get_sender_cumulative_features(
            sender=sender,
            current_time=dt,
            amount=amount,
        )

        receiver_features = self._get_receiver_cumulative_features(
            receiver=receiver,
            current_time=dt,
            amount=amount,
        )

        pair_features = self._get_pair_cumulative_features(
            sender=sender,
            receiver=receiver,
        )

        # 3. WINDOW FEATURES
        window_features = self._get_window_features(
            sender=sender,
            receiver=receiver,
            current_time=dt,
            amount=amount,
        )

        # 4. NUMERICAL TRANSACTION & DATETIME FEATURES
        transaction_features = {
            "Amount": amount,
            "log_amount": np.log1p(amount),
            "hour_sin": np.sin(2 * np.pi * dt.hour / 24),
            "hour_cos": np.cos(2 * np.pi * dt.hour / 24),
            "dow_sin": np.sin(2 * np.pi * dt.dayofweek / 7),
            "dow_cos": np.cos(2 * np.pi * dt.dayofweek / 7),
        }

        # COMBINE ALL FEATURES
        features = {
            **cat_features,
            **transaction_features,
            **sender_features,
            **receiver_features,
            **pair_features,
            **window_features,
        }

        return features

    # =====================================================
    # STATE UPDATES
    # =====================================================

    def update_cumulative_state(self, sender, receiver, amount, dt):
        # Sender Update
        if sender in self.sender_state.index:
            s_count = (
                self.sender_state.loc[sender, "sender_tx_count_history"] + 1
            )
            s_total = (
                self.sender_state.loc[sender, "sender_total_amount_history"]
                + amount
            )
            s_unique = self.sender_state.loc[
                sender, "sender_unique_receivers_history"
            ]
        else:
            s_count = 1
            s_total = amount
            s_unique = 0

        if sender not in self.sender_seen_receivers:
            self.sender_seen_receivers[sender] = set()

        if receiver not in self.sender_seen_receivers[sender]:
            self.sender_seen_receivers[sender].add(receiver)
            s_unique += 1

        self.sender_state.loc[sender] = {
            "sender_tx_count_history": s_count,
            "sender_total_amount_history": s_total,
            "sender_avg_amount_history": s_total / s_count,
            "sender_unique_receivers_history": s_unique,
            "sender_time_since_last_tx": dt,
        }

        # Receiver Update
        if receiver in self.receiver_state.index:
            r_count = (
                self.receiver_state.loc[receiver, "receiver_tx_count_history"]
                + 1
            )
            r_total = (
                self.receiver_state.loc[
                    receiver, "receiver_total_amount_history"
                ]
                + amount
            )
            r_unique = self.receiver_state.loc[
                receiver, "receiver_unique_senders_history"
            ]
        else:
            r_count = 1
            r_total = amount
            r_unique = 0

        if receiver not in self.receiver_seen_senders:
            self.receiver_seen_senders[receiver] = set()

        if sender not in self.receiver_seen_senders[receiver]:
            self.receiver_seen_senders[receiver].add(sender)
            r_unique += 1

        self.receiver_state.loc[receiver] = {
            "receiver_tx_count_history": r_count,
            "receiver_total_amount_history": r_total,
            "receiver_avg_amount_history": r_total / r_count,
            "receiver_unique_senders_history": r_unique,
            "receiver_time_since_last_tx": dt,
        }

        # Pair State Update
        pair_key = (sender, receiver)
        self.pair_state[pair_key] = self.pair_state.get(pair_key, 0) + 1

    def update_state(self, row):
        dt = pd.to_datetime(row["Datetime"])
        sender = row["Sender_account"]
        receiver = row["Receiver_account"]
        amount = float(row["Amount"])

        # Cumulative
        self.update_cumulative_state(
            sender=sender,
            receiver=receiver,
            amount=amount,
            dt=dt,
        )

        # Recent
        self.recent_state.add_transaction(
            entity_type="sender",
            entity_id=sender,
            dt=dt,
            amount=amount,
        )

        self.recent_state.add_transaction(
            entity_type="receiver",
            entity_id=receiver,
            dt=dt,
            amount=amount,
        )

    # =====================================================
    # PREDICTION METHOD
    # =====================================================

    def predict(self, new_transactions):
        df = new_transactions.copy()
        df["Datetime"] = pd.to_datetime(df["Datetime"])

        # Urutkan berdasarkan waktu transaksi (kronologis)
        df = df.sort_values("Datetime").reset_index(drop=True)

        predictions = []

        for _, row in df.iterrows():
            # 1. READ CURRENT STATE & BUILD FEATURES
            features = self.build_features_for_transaction(row)

            # 2. MODEL INPUT (Pastikan urutan kolom sesuai self.feature_cols)
            X = pd.DataFrame([features])
            if self.feature_cols:
                X = X[self.feature_cols]

            # 3. PREDICT (Gunakan self.pipeline dan self.threshold dari bundle)
            probability = self.pipeline.predict_proba(X)[0, 1]
            prediction = int(probability >= self.threshold)

            # 4. STORE RESULT
            result = row.to_dict()
            result["aml_probability"] = probability
            result["prediction"] = prediction

            # Untuk debugging / melacak fitur saat transaksi terjadi
            result.update(features)

            predictions.append(result)

            # 5. UPDATE STATE SETELAH PREDIKSI (Stateful Stream Update)
            self.update_state(row)

        return pd.DataFrame(predictions)

    def predict_batch(self, new_transactions, n_rows=10):
        """
        Stateful batched inference.
    
        Parameters
        ----------
        new_transactions : pd.DataFrame
            Dataset transaksi yang akan digunakan untuk inference.
            Biasanya test_df.
    
        n_rows : int
            Berapa banyak transaksi berikutnya yang akan diprediksi
            pada call ini.
    
        Behavior
        --------
        Call 1, n_rows=10
            -> row 1-10
    
        Call 2, n_rows=15
            -> row 11-25
    
        Call 3, n_rows=20
            -> row 26-45
    
        State akan tetap dipertahankan antar call.
        """
    
        # =====================================================
        # VALIDATION
        # =====================================================
    
        if n_rows <= 0:
            raise ValueError("n_rows harus lebih besar dari 0.")
    
        df = new_transactions.copy()
    
        df["Datetime"] = pd.to_datetime(df["Datetime"])
    
        # Pastikan kronologis
        df = (
            df
            .sort_values("Datetime")
            .reset_index(drop=True)
        )
    
        total_rows = len(df)
    
        # =====================================================
        # CHECK WHETHER DATASET IS FINISHED
        # =====================================================
    
        if self.inference_position >= total_rows:
            print("Semua transaksi pada test set sudah diprediksi.")
            return pd.DataFrame()
    
        # =====================================================
        # DETERMINE CURRENT BATCH
        # =====================================================
    
        start_idx = self.inference_position
        end_idx = min(
            start_idx + n_rows,
            total_rows
        )
    
        batch_df = df.iloc[start_idx:end_idx].copy()
    
        print(
            f"Predicting rows "
            f"{start_idx + 1} - {end_idx} "
            f"of {total_rows}"
        )
    
        # =====================================================
        # BUILD FEATURES SEQUENTIALLY
        # =====================================================
    
        feature_rows = []
        original_rows = []
    
        for _, row in batch_df.iterrows():
    
            # ---------------------------------------------
            # 1. READ CURRENT STATE
            # ---------------------------------------------
    
            features = self.build_features_for_transaction(row)
    
            feature_rows.append(features)
            original_rows.append(row.to_dict())
    
            # ---------------------------------------------
            # 2. UPDATE STATE
            # ---------------------------------------------
            #
            # Penting:
            # Row berikutnya harus bisa melihat transaksi ini.
            #
            # Jadi meskipun model prediction dilakukan secara
            # batch, feature generation tetap sequential.
            #
    
            self.update_state(row)
    
        # =====================================================
        # CREATE BATCH MODEL INPUT
        # =====================================================
    
        X = pd.DataFrame(feature_rows)
    
        if self.feature_cols:
            X = X[self.feature_cols]
    
        # =====================================================
        # BATCH PREDICTION
        # =====================================================
    
        probabilities = self.pipeline.predict_proba(X)[:, 1]
    
        predictions = (
            probabilities >= self.threshold
        ).astype(int)
    
        # =====================================================
        # BUILD OUTPUT
        # =====================================================
    
        results = []
    
        for i in range(len(batch_df)):
    
            result = original_rows[i]
    
            result["aml_probability"] = probabilities[i]
            result["prediction"] = predictions[i]
    
            # Untuk debugging / explainability
            result.update(feature_rows[i])
    
            results.append(result)
    
        results_df = pd.DataFrame(results)
    
        # =====================================================
        # MOVE CURSOR
        # =====================================================
    
        self.inference_position = end_idx
    
        # =====================================================
        # INFO
        # =====================================================
    
        print(
            f"Cursor sekarang: "
            f"{self.inference_position}/{total_rows}"
        )
    
        print(
            f"Sisa transaksi: "
            f"{total_rows - self.inference_position}"
        )
    
        return results_df

    def remaining_rows(self, new_transactions):
        df = new_transactions.copy()
    
        return max(
            len(df) - self.inference_position,
            0
        )
        

In [14]:
model_path = "/kaggle/input/notebooks/mrajareivan/model-training-and-evaluation/aml_xgb_model.pkl"

pipeline = AMLStatefulInference(
    model_path=model_path,

    sender_state=sender_state,
    receiver_state=receiver_state,

    sender_seen_receivers=sender_seen_receivers,
    receiver_seen_senders=receiver_seen_senders,

    recent_state=recent_state,
    pair_state=pair_state,
    categorical_features_dict=categorical_values
)

In [15]:
# results = pipeline.predict(
#        test_df
# )

In [16]:
# evaluation_df = results[
#     [
#         "Datetime",
#         "Sender_account",
#         "Receiver_account",
#         "Amount",
#         "Is_laundering",
#         "aml_probability",
#         "prediction",
#     ]
# ]

In [17]:
# evaluation_df

In [18]:
# from sklearn.metrics import classification_report

# print(
#     classification_report(
#         evaluation_df["Is_laundering"],
#         evaluation_df["prediction"]
#     )
# )

In [19]:
pipeline.info()

model threshold : 0.7500000000000001
model type : XGBoost


In [20]:
results_1 = pipeline.predict_batch(
    test_df,
    n_rows=10
)

Predicting rows 1 - 10 of 1425728
Cursor sekarang: 10/1425728
Sisa transaksi: 1425718


In [21]:
results_1.columns

Index(['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount',
       'Payment_currency', 'Received_currency', 'Sender_bank_location',
       'Receiver_bank_location', 'Payment_type', 'Is_laundering',
       'Laundering_type', 'Datetime', 'is_cross_border',
       'is_currency_conversion', 'hour', 'day_of_week', 'date_only',
       'day_of_month', 'aml_probability', 'prediction', 'log_amount',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'sender_tx_count_history',
       'sender_total_amount_history', 'sender_avg_amount_history',
       'sender_amount_vs_history_ratio', 'sender_unique_receivers_history',
       'sender_time_since_last_tx', 'receiver_tx_count_history',
       'receiver_total_amount_history', 'receiver_avg_amount_history',
       'receiver_amount_vs_history_ratio', 'receiver_unique_senders_history',
       'receiver_time_since_last_tx', 'pair_tx_count_history',
       'sender_tx_count_1h', 'sender_total_amount_1h', 'sender_avg_amount_1h',
       'sender_t

In [30]:
evaluation_df1 = results_1[
    [
        "Datetime",
        "Sender_account",
        "Receiver_account",
        "Amount",
        "Is_laundering",
        "aml_probability",
        "prediction",
    ]
]

In [31]:
evaluation_df1[evaluation_df1["Is_laundering"]==evaluation_df1["prediction"]]

,Datetime,Sender_account,Receiver_account,Amount,Is_laundering,aml_probability,prediction
0,2023-07-05 15:24:42,7850710817,7576305607,5058.98,0,6.395175e-07,0
1,2023-07-05 15:24:42,3337189497,3268922659,10000.80,0,6.262390e-08,0
2,2023-07-05 15:24:43,9952889104,9014196828,2418.34,0,3.516652e-06,0
3,2023-07-05 15:24:45,2266586338,3722009771,116.10,0,7.774963e-08,0
4,2023-07-05 15:24:48,8347362749,4208825830,4777.12,0,2.898720e-09,0
5,2023-07-05 15:24:48,366504489,7492584888,8510.80,0,1.300048e-08,0
6,2023-07-05 15:24:48,9669134064,1480239329,7324.79,0,3.407460e-08,0
7,2023-07-05 15:24:49,8264629738,3799029629,1188.85,0,9.365146e-05,0
8,2023-07-05 15:24:50,4570499134,2118974217,19644.59,0,8.835452e-07,0
9,2023-07-05 15:24:51,7545872000,505362441,10047.32,0,4.481250e-09,0


In [24]:
results_2 = pipeline.predict_batch(
    test_df,
    n_rows=13
)

Predicting rows 11 - 23 of 1425728
Cursor sekarang: 23/1425728
Sisa transaksi: 1425705


In [28]:
evaluation_df2 = results_2[
    [
        "Datetime",
        "Sender_account",
        "Receiver_account",
        "Amount",
        "Is_laundering",
        "aml_probability",
        "prediction",
    ]
]

In [29]:
evaluation_df2[evaluation_df2["Is_laundering"]==evaluation_df2["prediction"]]

,Datetime,Sender_account,Receiver_account,Amount,Is_laundering,aml_probability,prediction
0,2023-07-05 15:24:54,1697001278,3483875512,5097.86,0,1.156912e-07,0
1,2023-07-05 15:24:56,6602442353,6577810835,4962.41,0,1.593596e-09,0
2,2023-07-05 15:25:02,109069632,3227916128,4148.95,0,1.680391e-05,0
3,2023-07-05 15:25:07,751484310,3296333786,6076.41,0,3.457218e-06,0
4,2023-07-05 15:25:08,8910053718,4881672817,1717.26,0,2.704886e-05,0
5,2023-07-05 15:25:08,8968592383,2682194890,6108.27,0,1.814275e-08,0
6,2023-07-05 15:25:09,296807548,1066110007,2401.56,0,3.182250e-06,0
7,2023-07-05 15:25:12,4845524197,6593095600,14068.05,0,3.388845e-08,0
8,2023-07-05 15:25:13,51327985,1667716246,2182.66,0,5.712651e-06,0
9,2023-07-05 15:25:14,8159675205,5405399349,4041.28,0,1.470979e-05,0
